General tidy up of acquired Datasets:

In [52]:
import pandas as pd 
import matplotlib.pyplot as plt 
import gzip
import numpy as np

Helper Function for reading stuff:

In [2]:
def gzip_reader (path, usecols=None, dtype=str):
  return pd.read_csv(path, sep="\t", compression = "gzip", na_values = "\\N", dtype=dtype, usecols=usecols, low_memory=False)

Loading Basics:

In [3]:
print("Loading title.basics")
basics_cols = ["tconst", "titleType", "primaryTitle", "startYear", "runtimeMinutes", "genres"]
basics = gzip_reader('title.basics.tsv.gz', usecols=basics_cols)

Loading title.basics


In [4]:
basics.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,1894,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,1892,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,1892,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,1892,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,1893,1,Short


Loading akas:

In [ ]:
print("Loading title.akas...")
akas_cols = ['titleId','title','region','language','isOriginalTitle']
akas = gzip_reader('title.akas.tsv.gz', usecols=akas_cols)
akas.head(5)

Loading title.akas...


,titleId,title,region,language,isOriginalTitle
0,tt0000001,Carmencita,NaN,NaN,1
8,tt0000002,Le clown et ses chiens,NaN,NaN,1
16,tt0000003,Pauvre Pierrot,NaN,NaN,1
27,tt0000004,Un bon bock,NaN,NaN,1
35,tt0000005,Blacksmith Scene,NaN,NaN,1


Loading ratings:

In [6]:
print("Loading title.ratings")
ratings_cols = ["tconst", "averageRating", "numVotes"]
ratings = gzip_reader('title.ratings.tsv.gz', usecols=ratings_cols)
ratings.head()

Loading title.ratings


,tconst,averageRating,numVotes
0,tt0000001,5.7,2190
1,tt0000002,5.5,308
2,tt0000003,6.5,2289
3,tt0000004,5.1,196
4,tt0000005,6.2,3022


Joining Basics and Ratings based on tconst:

In [7]:
joint_table = pd.merge(basics,ratings, on="tconst",how="inner",validate="one_to_one")
joint_table.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2190
1,tt0000002,short,Le clown et ses chiens,1892,5,"Animation,Short",5.5,308
2,tt0000003,short,Poor Pierrot,1892,5,"Animation,Comedy,Romance",6.5,2289
3,tt0000004,short,Un bon bock,1892,12,"Animation,Short",5.1,196
4,tt0000005,short,Blacksmith Scene,1893,1,Short,6.2,3022


Joining Akas to the Joint Table:

In [23]:
merged_table = pd.merge(joint_table,akas,left_on="tconst",right_on="titleId",how="inner",validate="one_to_many").drop(columns=['titleId'])
merged_table.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle
0,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2190,Carmencita,NaN,NaN,1
1,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2190,Carmencita,DE,NaN,0
2,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2190,Carmencita,US,NaN,0
3,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2190,Carmencita - spanyol tánc,HU,NaN,0
4,tt0000001,short,Carmencita,1894,1,"Documentary,Short",5.7,2190,Καρμενσίτα,GR,NaN,0


Filtering By Bengali Films Only:

In [29]:
bengali_tags = merged_table[merged_table['language'] == 'bn']
bengali_tags.head(100)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle
39659,tt0018662,movie,Avalanche,1928,60,"Action,Drama,Romance",5.1,17,Junak prerije,BD,bn,0
231709,tt0042719,movie,Mashaal,1950,136,Drama,6.9,14,Saamar,IN,bn,0
263179,tt0045693,movie,Do Bigha Zamin,1953,131,Drama,8.3,2478,Do Bigha Zameen,IN,bn,0
278153,tt0047034,movie,Godzilla,1954,96,"Horror,Sci-Fi",7.6,44781,Gaḍajilā,BD,bn,0
282647,tt0047437,movie,Sabrina,1954,113,"Comedy,Drama,Romance",7.6,75213,Sabrina,IN,bn,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1348914,tt0217869,movie,Unbreakable,2000,106,"Drama,Mystery,Sci-Fi",7.3,464780,Unbreakable,IN,bn,0
1378969,tt0232168,movie,Naukadubi,1979,NaN,Drama,8.6,23,Noukadubi,IN,bn,0
1389461,tt0237254,movie,Fuleswari,1974,NaN,"Drama,Musical,Romance",8.5,39,Phuleswari,IN,bn,0
1389706,tt0237385,movie,Kothachilo,1994,NaN,Drama,9.0,13,Katha Chhilo,IN,bn,0


In [33]:
bengali_movies = bengali_tags[bengali_tags['titleType'] == 'movie']
bengali_movies['titleType'].value_counts().head(20)

titleType
movie    366
Name: count, dtype: int64

Filter out Hollywood translation films:

In [35]:
shady_mask = ~bengali_movies['genres'].str.contains(r'Adventure|Sci-Fi|SuperHero',case=False,na=False)
finalized_bengali_movies = bengali_movies[shady_mask]
finalized_bengali_movies.head(100)

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle
39659,tt0018662,movie,Avalanche,1928,60,"Action,Drama,Romance",5.1,17,Junak prerije,BD,bn,0
231709,tt0042719,movie,Mashaal,1950,136,Drama,6.9,14,Saamar,IN,bn,0
263179,tt0045693,movie,Do Bigha Zamin,1953,131,Drama,8.3,2478,Do Bigha Zameen,IN,bn,0
282647,tt0047437,movie,Sabrina,1954,113,"Comedy,Drama,Romance",7.6,75213,Sabrina,IN,bn,0
317107,tt0050484,movie,Harano Sur,1957,162,"Drama,Romance",8.2,2753,Harano Sur,IN,bn,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1667059,tt0364733,movie,Veedevadandi Babu,1997,138,"Action,Comedy",6.3,256,Amar Pratigya,IN,bn,0
1671291,tt0366304,movie,Choker Bali: A Passion Play,2003,167,Drama,7.2,2127,Chokher Bali,IN,bn,0
1672996,tt0366993,movie,Saanjhbatir Roopkathara,2002,120,"Drama,Family",7.2,164,Saanjhbatir Roopkathara,IN,bn,0
1678276,tt0368896,movie,Nijam,2003,196,"Action,Drama",6.6,2225,Amar Protishodh,IN,bn,0


Second Filter of English Lexicography:

In [40]:
english_pattern = r'\b(the|of|and|to|with|from|for|by|on|in|an)\b'
finalized_bengali_movies['english_grammar_signal'] = (
    finalized_bengali_movies['primaryTitle']
    .str.lower()
    .str.contains(english_pattern, na=False)
)
finalized_bengali_movies.head(20)


C:\Users\ARGHYA\AppData\Local\Temp\ipykernel_14740\842309846.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(english_pattern, na=False)
C:\Users\ARGHYA\AppData\Local\Temp\ipykernel_14740\842309846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  finalized_bengali_movies['english_grammar_signal'] = (


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle,english_grammar_signal
39659,tt0018662,movie,Avalanche,1928,60,"Action,Drama,Romance",5.1,17,Junak prerije,BD,bn,0,False
231709,tt0042719,movie,Mashaal,1950,136,Drama,6.9,14,Saamar,IN,bn,0,False
263179,tt0045693,movie,Do Bigha Zamin,1953,131,Drama,8.3,2478,Do Bigha Zameen,IN,bn,0,False
282647,tt0047437,movie,Sabrina,1954,113,"Comedy,Drama,Romance",7.6,75213,Sabrina,IN,bn,0,False
317107,tt0050484,movie,Harano Sur,1957,162,"Drama,Romance",8.2,2753,Harano Sur,IN,bn,0,False
327258,tt0051399,movie,Bari Theke Paliye,1958,124,Drama,8.0,287,Bari Theke Paliye,IN,bn,0,False
332164,tt0051792,movie,The Music Room,1958,95,"Drama,Music",7.8,7283,Jalsaghar,IN,bn,0,True
346840,tt0053104,movie,Neel Akasher Neechey,1959,133,Drama,7.6,160,Nil Akasher Niche,IN,bn,0,False
358415,tt0054073,movie,The Cloud-Capped Star,1960,126,Drama,7.8,3602,Meghe Dhaka Tara,IN,bn,0,True
377021,tt0055724,movie,The Expedition,1962,150,"Crime,Family,Romance",7.9,1424,Abhijan,IN,bn,0,True


In [44]:
finalized_bengali_movies['startYear'] = finalized_bengali_movies['startYear'].astype(int) 
finalized_bengali_movies['startYear'].value_counts().head(100)

C:\Users\ARGHYA\AppData\Local\Temp\ipykernel_14740\2844880654.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  finalized_bengali_movies['startYear'] = finalized_bengali_movies['startYear'].astype(int)


startYear
2025    16
2018    16
2017    13
2019    12
2024    12
        ..
2000     1
1950     1
1977     1
1931     1
1971     1
Name: count, Length: 75, dtype: int64

Filter Via Era's now again:

In [48]:
classics_era = finalized_bengali_movies[(finalized_bengali_movies['startYear'] >= 1970) & (finalized_bengali_movies['startYear'] <= 1995)]
modern_era = finalized_bengali_movies[(finalized_bengali_movies['startYear'] >= 2000) & (finalized_bengali_movies['startYear'] <= 2025)]
#classics_era.head(5)
#modern_era.head(5)

Final Prep work, Part 1: Era assignment

In [49]:
classics_era['era'] = "Classics"
modern_era['era'] = "Modern"

C:\Users\ARGHYA\AppData\Local\Temp\ipykernel_14740\4264550335.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  classics_era['era'] = "Classics"
C:\Users\ARGHYA\AppData\Local\Temp\ipykernel_14740\4264550335.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  modern_era['era'] = "Modern"


Combining two tables for a Principle Copy of processed Dataset:

In [54]:
combined_tags = pd.concat([classics_era, modern_era], ignore_index=True)
combined_tags['numVotes'] = combined_tags['numVotes'].astype(int)
combined_tags.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle,english_grammar_signal,era
0,tt0069204,movie,Sabse Bada Sukh,1972,114,"Comedy,Drama",6.0,21,Subse Bada Sukh,IN,bn,0,False,Classics
1,tt0070047,movie,The Exorcist,1973,122,Horror,8.1,493902,The Exorcist,IN,bn,0,True,Classics
2,tt0070047,movie,The Exorcist,1973,122,Horror,8.1,493902,Shoitan Raat,IN,bn,0,True,Classics
3,tt0073191,movie,Jai Santoshi Maa,1975,145,"Drama,Fantasy",6.6,242,Joy Sontoshi Maa,IN,bn,0,False,Classics
4,tt0078170,movie,Roads to the South,1978,100,Drama,5.5,291,Putevi na Jug,BD,bn,0,True,Classics


Taking Log of numVotes for reasons beyond my understanding(its 3am):

In [55]:
combined_tags['log_numVotes'] = np.log10(combined_tags['numVotes'] + 1)
combined_tags.head() 

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle,english_grammar_signal,era,log_numVotes
0,tt0069204,movie,Sabse Bada Sukh,1972,114,"Comedy,Drama",6.0,21,Subse Bada Sukh,IN,bn,0,False,Classics,1.342423
1,tt0070047,movie,The Exorcist,1973,122,Horror,8.1,493902,The Exorcist,IN,bn,0,True,Classics,5.693642
2,tt0070047,movie,The Exorcist,1973,122,Horror,8.1,493902,Shoitan Raat,IN,bn,0,True,Classics,5.693642
3,tt0073191,movie,Jai Santoshi Maa,1975,145,"Drama,Fantasy",6.6,242,Joy Sontoshi Maa,IN,bn,0,False,Classics,2.385606
4,tt0078170,movie,Roads to the South,1978,100,Drama,5.5,291,Putevi na Jug,BD,bn,0,True,Classics,2.465383


Saving Principal Copy:

In [56]:
combined_tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 269 entries, 0 to 268
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   tconst                  269 non-null    object 
 1   titleType               269 non-null    object 
 2   primaryTitle            269 non-null    object 
 3   startYear               269 non-null    int32  
 4   runtimeMinutes          238 non-null    object 
 5   genres                  269 non-null    object 
 6   averageRating           269 non-null    object 
 7   numVotes                269 non-null    int32  
 8   title                   269 non-null    object 
 9   region                  269 non-null    object 
 10  language                269 non-null    object 
 11  isOriginalTitle         269 non-null    object 
 12  english_grammar_signal  269 non-null    bool   
 13  era                     269 non-null    object 
 14  log_numVotes            269 non-null    fl

In [58]:
film_df = combined_tags
film_df.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,title,region,language,isOriginalTitle,english_grammar_signal,era,log_numVotes
0,tt0069204,movie,Sabse Bada Sukh,1972,114,"Comedy,Drama",6.0,21,Subse Bada Sukh,IN,bn,0,False,Classics,1.342423
1,tt0070047,movie,The Exorcist,1973,122,Horror,8.1,493902,The Exorcist,IN,bn,0,True,Classics,5.693642
2,tt0070047,movie,The Exorcist,1973,122,Horror,8.1,493902,Shoitan Raat,IN,bn,0,True,Classics,5.693642
3,tt0073191,movie,Jai Santoshi Maa,1975,145,"Drama,Fantasy",6.6,242,Joy Sontoshi Maa,IN,bn,0,False,Classics,2.385606
4,tt0078170,movie,Roads to the South,1978,100,Drama,5.5,291,Putevi na Jug,BD,bn,0,True,Classics,2.465383


In [59]:
film_df.to_csv("film_df.csv", index=False)